## TOPOAI — MULTI-RUN SUITE (5 LR SWEEPS) — FULL CORRECTED
**TOPO-2026 | Sovereign Machine Lab | Frank Morales Aguilera**

Runs 5 independent training passes over Tasks A → B → C, each with a different
learning-rate configuration. After all runs complete, metrics are averaged and
a single certified model (best Task-C accuracy) is pushed to the Hub.

In [ ]:
!nvidia-smi

Mon Jun  1 20:25:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
# ============================================================================
# UNIFIED TOPO-2026 MULTI-RUN SUITE — 5 LEARNING-RATE CONFIGURATIONS
# All other hyper-parameters (seed, data, primes) are held constant.
# After all runs: metrics are averaged; best checkpoint is pushed to the Hub.
# ============================================================================
import os
import gc
import copy
import time
import json
import shutil
import hashlib
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Dict, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from google.colab import userdata
from datasets import load_dataset
from huggingface_hub import login, HfApi, create_repo, upload_folder
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*torch_dtype.*')

# ============================================================================
# MULTI-RUN CONFIGURATION — edit this block only
# ============================================================================
NUM_RUNS   = 5          # total independent runs
FIXED_SEED = 123        # held constant across all runs
PRIME_LIMIT = 13        # anchor set size (held constant)
EPOCHS     = 6          # epochs per task per run
BATCH_SIZE = 16

# Learning-rate grid: (lr_embed, lr_cls) — one pair per run
LR_GRID = [
    (5e-3, 1e-3),   # Run 0 — original baseline
    (1e-3, 5e-4),   # Run 1 — lower embed LR
    (1e-2, 2e-3),   # Run 2 — higher embed LR
    (5e-3, 5e-3),   # Run 3 — equal LRs
    (2e-3, 1e-3),   # Run 4 — conservative embed
]
assert len(LR_GRID) == NUM_RUNS, "LR_GRID must have exactly NUM_RUNS entries."

# Hub publishing
YOUR_USERNAME  = 'frankmorales2020'
MODEL_NAME_HF  = 'topological-ai-gpt-oss-20b-multirun'
REPO_ID        = f'{YOUR_USERNAME}/{MODEL_NAME_HF}'
BASE_MODEL_ID  = 'openai/gpt-oss-20b'
HIDDEN_SIZE    = 2880
SAMPLE_A, SAMPLE_B, SAMPLE_C = 500, 1000, 1000


# ----------------------------------------------------------------------------
# 1. CORE ARCHITECTURE WRAPPERS
# ----------------------------------------------------------------------------
class GPT_OSS_20B_TaskAwareModel(nn.Module):
    """
    Task-Aware wrapper for GPT-OSS-20B.
    Freezes the transformer backbone; exposes 3 independent classification heads.
    """
    def __init__(self, base_model: nn.Module, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]
        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

    def freeze_previous_heads(self, task: str):
        if task == 'B':
            self.classifier_A.requires_grad_(False)
        elif task == 'C':
            self.classifier_B.requires_grad_(False)

    def reset_heads(self):
        """Re-initialise all three heads for a fresh run."""
        dev = next(self.base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        # Make all heads trainable again
        for h in (self.classifier_A, self.classifier_B, self.classifier_C):
            h.requires_grad_(True)
        self.current_task = 'A'


# ----------------------------------------------------------------------------
# 2. TOPOLOGICAL GOVERNOR ENGINE
# ----------------------------------------------------------------------------
class TopologicalGovernor:
    """Prime-anchored embedding constraint (Arithmetic Spectral Theory)."""
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]
        sieve = [True] * (prime_limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(prime_limit ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, prime_limit + 1, i):
                    sieve[j] = False
        primes = [i for i in range(2, prime_limit + 1) if sieve[i]]
        self.anchor_indices = [p for p in primes if p < vocab_size]
        self.snapshot = {}
        self.safety_constant = 1.0 - np.prod([1.0 - (p ** -0.5) for p in self.anchor_indices])

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )

    def get_hash(self) -> str:
        hasher = hashlib.sha256()
        for idx in self.anchor_indices:
            weight_bytes = self.embed_layer.weight[idx].detach().float().cpu().numpy().tobytes()
            hasher.update(weight_bytes)
        return hasher.hexdigest()[:16]


# ----------------------------------------------------------------------------
# 3. DATASET UTILITIES
# ----------------------------------------------------------------------------
class AGNewsStreamDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids    = input_ids
        self.attention_mask = attention_mask
        self.labels       = labels

    def __len__(self):  return len(self.labels)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'labels': self.labels[idx]}


def prepare_tokenized_dataset(tokenizer, texts, labels) -> AGNewsStreamDataset:
    tokens = tokenizer(texts, max_length=64, padding='max_length',
                       truncation=True, return_tensors='pt')
    return AGNewsStreamDataset(tokens.input_ids, tokens.attention_mask,
                               torch.tensor(labels, dtype=torch.long))


# ----------------------------------------------------------------------------
# 4. TRAINING & EVALUATION
# ----------------------------------------------------------------------------
def train_task_explicit(
    task_label: str,
    model: GPT_OSS_20B_TaskAwareModel,
    dataset: AGNewsStreamDataset,
    embed_layer: nn.Embedding,
    governor: Optional[TopologicalGovernor] = None,
    epochs: int = EPOCHS,
    batch_size: int = BATCH_SIZE,
    lr_embed: float = 5e-3,
    lr_cls: float   = 1e-3,
    run_id: int = 0
) -> float:
    model.switch_task(task_label)
    model.train()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    active_head = getattr(model, f'classifier_{task_label}')
    optimizer = torch.optim.AdamW([
        {'params': embed_layer.weight, 'lr': lr_embed},
        {'params': active_head.parameters(), 'lr': lr_cls}
    ])
    total_steps = epochs * len(dataloader)
    desc = f'[Run {run_id}] Task {task_label} | lr_embed={lr_embed:.0e} lr_cls={lr_cls:.0e}'
    progress_bar = tqdm(total=total_steps, desc=desc, leave=True)
    device = next(model.parameters()).device

    for epoch in range(epochs):
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss   = F.cross_entropy(logits, labels)
            loss.backward()
            if governor:
                governor.zero_anchor_gradients()
            torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
            optimizer.step()
            if governor:
                governor.enforce_anchors()
            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            progress_bar.update(1)

    progress_bar.close()
    return evaluate_model_precision(model, dataloader)


def evaluate_model_precision(model: GPT_OSS_20B_TaskAwareModel,
                              dataloader: DataLoader) -> float:
    model.eval()
    correct = total = 0
    device  = next(model.parameters()).device
    with torch.no_grad():
        for batch in dataloader:
            logits = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device)
            )
            preds    = torch.argmax(logits, dim=-1)
            correct += (preds == batch['labels'].to(device)).sum().item()
            total   += batch['labels'].size(0)
    return float(correct / total)


# ============================================================================
# 5. GLOBAL SETUP  (runs once — backbone is never re-loaded between runs)
# ============================================================================
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def full_vram_purge(objects_to_delete=None, sleep_secs=5):
    if objects_to_delete:
        for obj in objects_to_delete:
            if obj is not None:
                try:
                    if isinstance(obj, nn.Module):
                        obj.cpu()
                        for p in obj.parameters():
                            if p.grad is not None:
                                p.grad = None
                    del obj
                except:
                    pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    time.sleep(sleep_secs)


set_seed(FIXED_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=' * 75)
print(f'TOPO-2026 MULTI-RUN INITIALISATION  ({NUM_RUNS} runs, seed={FIXED_SEED})')
print('=' * 75)

# --- Dataset (loaded once) ---
print('\n[DATASET] Loading AG News splits...')
raw_ag_dataset = load_dataset('SetFit/ag_news', split='train')

def isolate_task_domain(dataset, class_labels, sample_limit):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled  = filtered.select(range(min(sample_limit, len(filtered))))
    texts  = [item['text']  for item in sampled]
    labels = [item['label'] % 2 for item in sampled]
    return texts, labels

task_a_texts, task_a_labels = isolate_task_domain(raw_ag_dataset, [0, 1], SAMPLE_A)
task_b_texts, task_b_labels = isolate_task_domain(raw_ag_dataset, [2, 3], SAMPLE_B)
task_c_texts, task_c_labels = isolate_task_domain(raw_ag_dataset, [0, 3], SAMPLE_C)

# FIX (Bug 3): held-out val splits from the AG News test split
print('[DATASET] Loading AG News test split for held-out val evaluation...')
raw_ag_test = load_dataset('SetFit/ag_news', split='test')
val_a_texts, val_a_labels = isolate_task_domain(raw_ag_test, [0, 1], 200)
val_b_texts, val_b_labels = isolate_task_domain(raw_ag_test, [2, 3], 200)
val_c_texts, val_c_labels = isolate_task_domain(raw_ag_test, [0, 3], 200)

# --- Backbone (loaded once, frozen once) ---
print('\n[BACKBONE] Materialising GPT-OSS-20B...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16
).to(device)
for param in base_model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# --- Locate embedding layer ---
if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
    embed_layer = base_model.transformer.wte
else:
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100000:
            embed_layer = module
            break
embed_layer.weight.requires_grad = True

# --- Pre-tokenise (once) ---
dataset_A = prepare_tokenized_dataset(tokenizer, task_a_texts, task_a_labels)
dataset_B = prepare_tokenized_dataset(tokenizer, task_b_texts, task_b_labels)
dataset_C = prepare_tokenized_dataset(tokenizer, task_c_texts, task_c_labels)

# Held-out val datasets (tokenised once, reused across all runs)
val_dataset_A = prepare_tokenized_dataset(tokenizer, val_a_texts, val_a_labels)
val_dataset_B = prepare_tokenized_dataset(tokenizer, val_b_texts, val_b_labels)
val_dataset_C = prepare_tokenized_dataset(tokenizer, val_c_texts, val_c_labels)

# --- Shared task-aware wrapper ---
model = GPT_OSS_20B_TaskAwareModel(base_model=base_model)

# --- Snapshot original embedding weights (restored before every run) ---
original_embed_weights = embed_layer.weight.detach().clone()

print('\n[READY] Setup complete. Starting multi-run sweep.')


TOPO-2026 MULTI-RUN INITIALISATION  (5 runs, seed=123)

[DATASET] Loading AG News splits...


train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

...

tokenizer_config.json:   0%|          | 0.00/4.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.7k [00:00<?, ?B/s]


[READY] Setup complete. Starting multi-run sweep.


In [ ]:
# ============================================================================
# 6. MULTI-RUN SWEEP LOOP
# ============================================================================
run_results: List[Dict] = []   # collects per-run metric dicts
best_run_idx    = -1
best_acc_c      = -1.0
best_state_dict = None

for run_id in range(NUM_RUNS):
    lr_embed, lr_cls = LR_GRID[run_id]

    print('\n' + '=' * 75)
    print(f'  RUN {run_id + 1}/{NUM_RUNS}  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}')
    print('=' * 75)

    # --- Reset per-run state ---
    set_seed(FIXED_SEED)
    model.reset_heads()
    with torch.no_grad():
        embed_layer.weight.copy_(original_embed_weights)

    # ------------------------------------------------------------------
    # TASK A
    # ------------------------------------------------------------------
    print(f'\n[RUN {run_id}] TASK A: World vs Sports')
    train_task_explicit(
        'A', model, dataset_A, embed_layer,
        governor=None, lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id
    )
    # Baseline: evaluate on TRAINING data right after Task A — same split used for final
    _dl_train_a = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
    acc_a_initial = evaluate_model_precision(model, _dl_train_a)
    print(f'  [TASK A] Train Baseline: {acc_a_initial * 100:.2f}%')

    # Memory consolidation snapshot
    governor = TopologicalGovernor(embed_layer=embed_layer, prime_limit=PRIME_LIMIT)
    print(f'  [HIPPOCAMPUS] Anchoring {len(governor.anchor_indices)} prime coords: {governor.anchor_indices}')
    t0 = time.perf_counter()
    governor.take_snapshot()
    print(f'  [HIPPOCAMPUS] Snapshot in {(time.perf_counter()-t0)*1000:.2f} ms | hash={governor.get_hash()}')
    print(f'  [HIPPOCAMPUS] Safety Constant Λ (AST): {governor.safety_constant:.10f}')

    # FIX (Bug 2): freeze A head BEFORE Task B optimizer is constructed
    model.freeze_previous_heads('B')

    # ------------------------------------------------------------------
    # TASK B
    # ------------------------------------------------------------------
    print(f'\n[RUN {run_id}] TASK B: Business vs Sci/Tech')
    train_task_explicit(
        'B', model, dataset_B, embed_layer,
        governor=governor, lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id
    )
    # Baseline: evaluate on TRAINING data right after Task B — same split used for final
    _dl_train_b = DataLoader(dataset_B, batch_size=BATCH_SIZE, shuffle=False)
    acc_b_initial = evaluate_model_precision(model, _dl_train_b)
    print(f'  [TASK B] Train Baseline: {acc_b_initial * 100:.2f}%')

    # FIX (Bug 2): freeze B head BEFORE Task C optimizer is constructed
    model.freeze_previous_heads('C')

    # ------------------------------------------------------------------
    # TASK C
    # ------------------------------------------------------------------
    print(f'\n[RUN {run_id}] TASK C: World vs Sci/Tech')
    train_task_explicit(
        'C', model, dataset_C, embed_layer,
        governor=governor, lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id
    )
    # Evaluate Task C on held-out val split
    _dl_val_c = DataLoader(val_dataset_C, batch_size=BATCH_SIZE, shuffle=False)
    acc_c_final = evaluate_model_precision(model, _dl_val_c)
    print(f'  [TASK C] Val Accuracy: {acc_c_final * 100:.2f}%')

    assert governor.verify_integrity(), f'[RUN {run_id}] Topological integrity violated!'

    # ------------------------------------------------------------------
    # POST-RUN FORGETTING MEASUREMENT
    # ------------------------------------------------------------------
    # Forgetting: re-evaluate on SAME training data as baseline — consistent with reference
    print(f'\n[RUN {run_id}] Measuring retention on training data...')
    dl_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
    dl_B = DataLoader(dataset_B, batch_size=BATCH_SIZE, shuffle=False)

    model.switch_task('A')
    acc_a_final = evaluate_model_precision(model, dl_A)
    print(f'  [TASK A] Final Train Accuracy: {acc_a_final * 100:.2f}%')

    model.switch_task('B')
    acc_b_final = evaluate_model_precision(model, dl_B)
    print(f'  [TASK B] Final Train Accuracy: {acc_b_final * 100:.2f}%')

    # Forgetting = train_baseline (right after task) - train_final (after all tasks)
    # Same data split on both sides — consistent with TOPO_HF reference implementation
    # Positive = forgot (got worse), Negative = backward transfer (got better)
    fgt_A        = (acc_a_initial - acc_a_final) * 100
    fgt_B        = (acc_b_initial - acc_b_final) * 100
    combined_fgt = (fgt_A + fgt_B) / 2.0
    anchor_kb    = (len(governor.anchor_indices) * embed_layer.weight.shape[1] * 4) / 1024

    run_record = {
        'run_id':        run_id,
        'lr_embed':      lr_embed,
        'lr_cls':        lr_cls,
        'acc_a_final':   acc_a_final,
        'acc_b_final':   acc_b_final,
        'acc_c_final':   acc_c_final,
        'fgt_A':         fgt_A,
        'fgt_B':         fgt_B,
        'combined_fgt':  combined_fgt,
        'anchor_kb':     anchor_kb,
        'anchor_hash':   governor.get_hash(),
    }
    run_results.append(run_record)

    # Per-run summary box
    print(f'\n  ┌──────────────────────────────────────────────────────────────┐')
    print(f'  │  RUN {run_id} SUMMARY  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}')
    print(f'  ├──────────────────────────────────────────────────────────────┤')
    print(f'  │  Task A  acc={acc_a_final*100:6.2f}%  fgt={fgt_A:+6.2f}%  (World vs Sports)')
    print(f'  │  Task B  acc={acc_b_final*100:6.2f}%  fgt={fgt_B:+6.2f}%  (Business vs Sci/Tech)')
    print(f'  │  Task C  acc={acc_c_final*100:6.2f}%            (World vs Sci/Tech)')
    print(f'  │  Combined Forgetting : {combined_fgt:+.2f}%')
    print(f'  │  Anchor Memory       : {anchor_kb:.2f} KB')
    print(f'  └──────────────────────────────────────────────────────────────┘')

    # Track best model by Task C accuracy
    # Save state dict on CPU to avoid OOM during deepcopy on GPU
    if acc_c_final > best_acc_c:
        best_acc_c   = acc_c_final
        best_run_idx = run_id
        cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
        best_state_dict = copy.deepcopy(cpu_state)
        del cpu_state
        print(f'  ★ New best model saved (Run {run_id}, Task C: {acc_c_final*100:.2f}%)')

    # ------------------------------------------------------------------
    # END-OF-RUN MEMORY PURGE
    # Explicitly delete ALL per-run transient objects before calling
    # full_vram_purge so their GPU memory is returned to the allocator.
    # Objects that must NOT be deleted: model, base_model, embed_layer,
    # original_embed_weights, dataset_*, val_dataset_*, tokenizer.
    # ------------------------------------------------------------------
    print(f'\n[RUN {run_id}] Purging GPU memory...')

    # 1. Zero embedding gradient buffer explicitly
    if embed_layer.weight.grad is not None:
        embed_layer.weight.grad = None

    # 2. Clear governor snapshot (CPU tensors but still referenced)
    if governor is not None:
        governor.snapshot.clear()

    # 3. Delete all per-run transient dataloaders and temp vars
    for _obj in [governor, dl_A, dl_B, _dl_train_a, _dl_train_b, _dl_val_c]:
        try:
            del _obj
        except Exception:
            pass

    # 4. Full CUDA purge with sleep
    full_vram_purge(objects_to_delete=None)

    if torch.cuda.is_available():
        alloc_gb  = torch.cuda.memory_allocated() / 1024**3
        reserv_gb = torch.cuda.memory_reserved()  / 1024**3
        print(f'  [PURGE] VRAM → allocated: {alloc_gb:.3f} GB | reserved: {reserv_gb:.3f} GB')

print('\n' + '=' * 75)
print('ALL RUNS COMPLETE')
print('=' * 75)



  RUN 1/5  |  lr_embed=5e-03  lr_cls=1e-03

[RUN 0] TASK A: World vs Sports


[Run 0] Task A | lr_embed=5e-03 lr_cls=1e-03:   0%|          | 0/192 [00:00<?, ?it/s]

  [TASK A] Train Baseline: 99.80%
  [HIPPOCAMPUS] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [HIPPOCAMPUS] Snapshot in 0.21 ms | hash=9b506d27c06a1864
  [HIPPOCAMPUS] Safety Constant Λ (AST): 0.9785142874

[RUN 0] TASK B: Business vs Sci/Tech


[Run 0] Task B | lr_embed=5e-03 lr_cls=1e-03:   0%|          | 0/378 [00:00<?, ?it/s]

  [TASK B] Train Baseline: 99.70%

[RUN 0] TASK C: World vs Sci/Tech


...

  [TASK A] Train Baseline: 98.40%
  [HIPPOCAMPUS] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [HIPPOCAMPUS] Snapshot in 0.22 ms | hash=e30f40a19b4b23e3
  [HIPPOCAMPUS] Safety Constant Λ (AST): 0.9785142874

[RUN 4] TASK B: Business vs Sci/Tech


[Run 4] Task B | lr_embed=2e-03 lr_cls=1e-03:   0%|          | 0/378 [00:00<?, ?it/s]

  [TASK B] Train Baseline: 100.00%

[RUN 4] TASK C: World vs Sci/Tech


[Run 4] Task C | lr_embed=2e-03 lr_cls=1e-03:   0%|          | 0/378 [00:00<?, ?it/s]

  [TASK C] Val Accuracy: 91.50%

[RUN 4] Measuring retention on training data...
  [TASK A] Final Train Accuracy: 97.40%
  [TASK B] Final Train Accuracy: 99.70%

  ┌──────────────────────────────────────────────────────────────┐
  │  RUN 4 SUMMARY  |  lr_embed=2e-03  lr_cls=1e-03
  ├──────────────────────────────────────────────────────────────┤
  │  Task A  acc= 97.40%  fgt= +1.00%  (World vs Sports)
  │  Task B  acc= 99.70%  fgt= +0.30%  (Business vs Sci/Tech)
  │  Task C  acc= 91.50%            (World vs Sci/Tech)
  │  Combined Forgetting : +0.65%
  │  Anchor Memory       : 67.50 KB
  └──────────────────────────────────────────────────────────────┘

[RUN 4] Purging GPU memory...
  [PURGE] VRAM → allocated: 40.055 GB | reserved: 40.234 GB

ALL RUNS COMPLETE


In [ ]:
# ============================================================================
# 7. AGGREGATE METRICS
# ============================================================================
import statistics

avg_acc_c     = statistics.mean(r['acc_c_final']  for r in run_results)
std_acc_c     = statistics.stdev(r['acc_c_final'] for r in run_results) if NUM_RUNS > 1 else 0.0
avg_fgt       = statistics.mean(r['combined_fgt'] for r in run_results)
std_fgt       = statistics.stdev(r['combined_fgt'] for r in run_results) if NUM_RUNS > 1 else 0.0
avg_anchor_kb = statistics.mean(r['anchor_kb']    for r in run_results)

print('\n' + '=' * 75)
print('COMPILING MULTI-RUN EMPIRICAL PERFORMANCE MATRIX')
print('=' * 75)
print(f'{'Run':>4}  {'lr_embed':>10}  {'lr_cls':>8}  '
      f'{'Acc_A':>7}  {'Acc_B':>7}  {'Acc_C':>7}  {'Fgt':>8}')
print('-' * 75)
for r in run_results:
    marker = ' ★' if r['run_id'] == best_run_idx else ''
    print(f"{r['run_id']:>4}  {r['lr_embed']:>10.0e}  {r['lr_cls']:>8.0e}  "
          f"{r['acc_a_final']*100:>6.2f}%  {r['acc_b_final']*100:>6.2f}%  "
          f"{r['acc_c_final']*100:>6.2f}%  {r['combined_fgt']:>+7.2f}%{marker}")
print('-' * 75)
print(f"{'MEAN':>4}  {'':>10}  {'':>8}  "
      f"{'':>7}  {'':>7}  {avg_acc_c*100:>6.2f}%  {avg_fgt:>+7.2f}%")
print(f"{'STD':>4}  {'':>10}  {'':>8}  "
      f"{'':>7}  {'':>7}  {std_acc_c*100:>6.2f}%  {std_fgt:>+7.2f}%")
print('=' * 75)

cert_task_c = 'PASS' if avg_acc_c * 100 >= 85.0 else 'FAIL'
cert_fgt    = 'PASS' if avg_fgt <= 10.0        else 'FAIL'
print(f'\nTOPO-2026 CERTIFICATION (averaged over {NUM_RUNS} runs)')
print(f'  Task C accuracy : {avg_acc_c*100:.1f}% ± {std_acc_c*100:.1f}%  (threshold ≥85%) → {cert_task_c}')
print(f'  Combined fgt    : {avg_fgt:.1f}% ± {std_fgt:.1f}%  (threshold ≤10%) → {cert_fgt}')
print(f'  Best run        : Run {best_run_idx}  (lr_embed={LR_GRID[best_run_idx][0]:.0e}, lr_cls={LR_GRID[best_run_idx][1]:.0e})')



COMPILING MULTI-RUN EMPIRICAL PERFORMANCE MATRIX
 Run    lr_embed    lr_cls    Acc_A    Acc_B    Acc_C       Fgt
---------------------------------------------------------------------------
   0       5e-03     1e-03   98.40%   97.40%   93.50%    +1.85%
   1       1e-03     5e-04   99.60%   99.90%   89.50%    -0.05%
   2       1e-02     2e-03   92.40%   99.30%   92.50%    +3.25%
   3       5e-03     5e-03   96.60%   98.30%   94.50%    +2.05% ★
   4       2e-03     1e-03   97.40%   99.70%   91.50%    +0.65%
---------------------------------------------------------------------------
MEAN                                           92.30%    +1.55%
 STD                                            1.92%    +1.28%

TOPO-2026 CERTIFICATION (averaged over 5 runs)
  Task C accuracy : 92.3% ± 1.9%  (threshold ≥85%) → PASS
  Combined fgt    : 1.6% ± 1.3%  (threshold ≤10%) → PASS
  Best run        : Run 3  (lr_embed=5e-03, lr_cls=5e-03)


In [ ]:
# ============================================================================
# 8. LOCAL SERIALISATION  (best checkpoint)
# ============================================================================
LOCAL_PATH = './topological_ai_multirun_certified'
os.makedirs(LOCAL_PATH, exist_ok=True)

torch.save(best_state_dict, f'{LOCAL_PATH}/certified_topological_best.pt')
tokenizer.save_pretrained(LOCAL_PATH)

config_payload = {
    'certification_standard': 'TOPO-2026-MULTIRUN',
    'num_runs': NUM_RUNS,
    'fixed_seed': FIXED_SEED,
    'lr_grid': LR_GRID,
    'best_run': {
        'run_id':    best_run_idx,
        'lr_embed':  LR_GRID[best_run_idx][0],
        'lr_cls':    LR_GRID[best_run_idx][1],
        'acc_c':     f'{best_acc_c*100:.1f}%',
    },
    'aggregated': {
        'task_c_accuracy_mean': f'{avg_acc_c*100:.1f}%',
        'task_c_accuracy_std':  f'{std_acc_c*100:.1f}%',
        'task_c_threshold':     '>=85%',
        'task_c_status':        cert_task_c,
        'combined_forgetting_mean': f'{avg_fgt:.1f}%',
        'combined_forgetting_std':  f'{std_fgt:.1f}%',
        'forgetting_threshold': '<=10%',
        'forgetting_status':    cert_fgt,
        'anchor_memory_kb':     f'{avg_anchor_kb:.2f}',
        'anchor_integrity':     'PASS',
    },
    'all_runs': run_results,
    'prime_limit':   PRIME_LIMIT,
    'prime_anchors': [2, 3, 5, 7, 11, 13],
    'safety_constant': float(1.0 - np.prod([1.0-(p**-0.5) for p in [2,3,5,7,11,13]])),
    'base_model':    BASE_MODEL_ID,
}
with open(f'{LOCAL_PATH}/topological_config.json', 'w') as f:
    json.dump(config_payload, f, indent=2)

# Per-run results CSV for quick inspection
import csv
csv_path = f'{LOCAL_PATH}/run_results.csv'
fieldnames = list(run_results[0].keys())
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(run_results)
print(f'✓ Per-run CSV saved: {csv_path}')

# README
best = run_results[best_run_idx]
rows = '\n'.join(
    f"| Run {r['run_id']} | {r['lr_embed']:.0e} / {r['lr_cls']:.0e} | "
    f"{r['acc_a_final']*100:.2f}% | {r['acc_b_final']*100:.2f}% | "
    f"{r['acc_c_final']*100:.2f}% | {r['combined_fgt']:+.2f}% |"
    for r in run_results
)
readme = f"""---
license: apache-2.0
tags:
  - continual-learning
  - catastrophic-forgetting
  - topological-ai
  - TOPO-2026
base_model: {BASE_MODEL_ID}
---

# Topological AI — GPT-OSS-20B Multi-Run (TOPO-2026 Certified)

**Author:** Frank Morales Aguilera, BEng, MEng, SMIEEE
**Lab:** Sovereign Machine Lab (SOMALA), Montréal, Canada
**Paper:** https://zenodo.org/records/20338459

## Multi-Run Configuration
- **Runs:** {NUM_RUNS} (varying `lr_embed` / `lr_cls`; seed={FIXED_SEED} held constant)
- **Deployed checkpoint:** Best Task-C run (Run {best_run_idx})

## TOPO-2026 Track II Certificate (averaged over {NUM_RUNS} runs)
| Metric | Mean ± Std | Threshold | Status |
| --- | --- | --- | --- |
| Task C Accuracy | {avg_acc_c*100:.1f}% ± {std_acc_c*100:.1f}% | ≥85% | **{cert_task_c}** |
| Combined Forgetting | {avg_fgt:.1f}% ± {std_fgt:.1f}% | ≤10% | **{cert_fgt}** |
| Anchor Memory | {avg_anchor_kb:.2f} KB | O(1) | **PASS** |

## Per-Run Empirical Ledger
| Run | lr_embed / lr_cls | Acc A | Acc B | Acc C | Forgetting |
| --- | --- | --- | --- | --- | --- |
{rows}
"""
with open(f'{LOCAL_PATH}/README.md', 'w') as f:
    f.write(readme)
print(f'✓ Assets staged at: {LOCAL_PATH}')


✓ Per-run CSV saved: ./topological_ai_multirun_certified/run_results.csv
✓ Assets staged at: ./topological_ai_multirun_certified


In [ ]:
# ============================================================================
# 9. HUB PUSH  (single upload of best checkpoint + aggregated report)
# ============================================================================
print('\n[AUTH] Requesting HF_TOKEN...')
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=True)
except Exception as e:
    print(f'UserData hook skipped: {e}')
    login(add_to_git_credential=True)
    HF_TOKEN = None

try:
    create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True,
                private=False, token=HF_TOKEN)
    print(f'✓ Repository ready: {REPO_ID}')
except Exception as e:
    print(f'Repository setup note: {e}')

commit_msg = (
    f'TOPO-2026 Multi-Run | {NUM_RUNS} runs | '
    f'Best Run {best_run_idx} | '
    f'Avg Task-C: {avg_acc_c*100:.1f}% | '
    f'Avg Fgt: {avg_fgt:.1f}% | '
    f'Λ={float(1.0 - np.prod([1.0-(p**-0.5) for p in [2,3,5,7,11,13]])):.10f} | '
    f'Anchors: [2,3,5,7,11,13]'
)
print(f'\n🚀 Uploading to {REPO_ID}...')
upload_folder(
    repo_id=REPO_ID,
    folder_path=LOCAL_PATH,
    repo_type='model',
    token=HF_TOKEN,
    commit_message=commit_msg
)
print(f'✨ Deployment complete → https://huggingface.co/{REPO_ID}')



[AUTH] Requesting HF_TOKEN...
✓ Repository ready: frankmorales2020/topological-ai-gpt-oss-20b-multirun

🚀 Uploading to frankmorales2020/topological-ai-gpt-oss-20b-multirun...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ified_topological_best.pt:   0%|          | 88.0MB / 41.8GB            

  ..._certified/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

✨ Deployment complete → https://huggingface.co/frankmorales2020/topological-ai-gpt-oss-20b-multirun


In [ ]:
# ============================================================================
# 10. INFERENCE VERIFICATION  (best checkpoint, Task C spot-check)
# ============================================================================
from huggingface_hub import hf_hub_download

print('=' * 75)
print('TOPO-2026 MULTI-RUN INFERENCE VERIFICATION')
print('=' * 75)

certified_weights_path = hf_hub_download(
    repo_id=REPO_ID, filename='certified_topological_best.pt'
)

# Re-use the already-loaded backbone; just re-initialise heads and reload weights
model.reset_heads()
state_dict = torch.load(certified_weights_path, map_location='cpu')
model.load_state_dict(state_dict, strict=True)
model.eval()
print('✓ Best checkpoint loaded.')

test_phrase = 'New quantum computing startup secures massive initial funding round for enterprise deployment.'
inputs = tokenizer(test_phrase, return_tensors='pt').to(device)

with torch.no_grad():
    model.switch_task('C')
    logits_C = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
    probs_C  = F.softmax(logits_C.float(), dim=-1).squeeze().cpu().numpy()

pred_class  = int(np.argmax(probs_C))
confidence  = float(probs_C[pred_class])

print(f'\nInput  : "{test_phrase}"')
print(f'Task C output vector : {probs_C}')
print(f'Predicted class      : Label {pred_class}')
print(f'Confidence           : {confidence*100:.2f}%')

if confidence >= 0.85:
    print(f'>>> TOPO-2026 CERTIFIED ✓  Confidence {confidence*100:.2f}% meets production threshold (≥85%).')
elif confidence >= 0.70:
    print(f'>>> PASS  Confidence {confidence*100:.2f}% within acceptable range.')
else:
    print(f'>>> NOTICE  Confidence {confidence*100:.2f}% below expected range — check data distribution.')

print('\n' + '=' * 75)
print('MULTI-RUN INFERENCE TEST COMPLETE')
print('=' * 75)


TOPO-2026 MULTI-RUN INFERENCE VERIFICATION


certified_topological_best.pt:   0%|          | 0.00/41.8G [00:00<?, ?B/s]

✓ Best checkpoint loaded.

Input  : "New quantum computing startup secures massive initial funding round for enterprise deployment."
Task C output vector : [5.976847e-34 1.000000e+00]
Predicted class      : Label 1
Confidence           : 100.00%
>>> TOPO-2026 CERTIFIED ✓  Confidence 100.00% meets production threshold (≥85%).

MULTI-RUN INFERENCE TEST COMPLETE


In [ ]:
# ============================================================================
# 11. STANDALONE HF INFERENCE — run independently, no prior cells needed
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, math
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download

HF_REPO_ID    = 'frankmorales2020/topological-ai-gpt-oss-20b-multirun'
BASE_MODEL_ID = 'openai/gpt-oss-20b'
HIDDEN_SIZE   = 2880
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TASK_LABELS   = {'A':{0:'World',1:'Sports'},'B':{0:'Business',1:'Sci/Tech'},'C':{0:'World',1:'Sci/Tech'}}
TEST_INPUTS   = [
    ('A', 'The national team won the championship after a stunning comeback victory.'),
    ('B', 'Quarterly earnings beat analyst expectations driven by strong cloud revenue growth.'),
    ('C', 'New quantum computing startup secures massive initial funding round for enterprise deployment.'),
]

class TaskAwareModel(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base_model   = base
        dev = next(base.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'
    def forward(self, input_ids, attention_mask=None):
        h = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        if attention_mask is not None:
            h = h[torch.arange(input_ids.shape[0], device=input_ids.device), torch.eq(attention_mask,1).int().sum(-1)-1, :]
        else:
            h = h[:,-1,:]
        return getattr(self, f'classifier_{self.current_task}')(h)
    def switch_task(self, t): self.current_task = t

print('='*75)
print(f'TOPO-2026 HF INFERENCE  |  {HF_REPO_ID}')
print(f'Safety Constant Λ: {1.0 - math.prod(1.0-p**-0.5 for p in [2,3,5,7,11,13]):.10f}')
print('='*75)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16).to(device)
for p in base.parameters(): p.requires_grad = False
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tok.pad_token = tok.eos_token
m = TaskAwareModel(base)
m.load_state_dict(torch.load(hf_hub_download(repo_id=HF_REPO_ID, filename='certified_topological_best.pt'), map_location='cpu'), strict=False)
m.eval()
print('✓ Certified checkpoint loaded from Hub\n')
for task, sentence in TEST_INPUTS:
    inp = tok(sentence, return_tensors='pt', max_length=64, padding='max_length', truncation=True).to(device)
    with torch.no_grad():
        m.switch_task(task)
        probs = F.softmax(m(inp.input_ids, inp.attention_mask).float(), dim=-1).squeeze().cpu().numpy()
    idx, conf = int(np.argmax(probs)), float(probs.max())
    status = '✓ CERTIFIED' if conf >= 0.85 else '~ PASS' if conf >= 0.70 else '✗ LOW'
    print(f'Task {task} [{status}]  {TASK_LABELS[task][idx]:10s}  {conf*100:.2f}%  "{sentence[:65]}"')
print('\n'+'='*75)


TOPO-2026 HF INFERENCE  |  frankmorales2020/topological-ai-gpt-oss-20b-multirun
Safety Constant Λ: 0.9785142874


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

✓ Certified checkpoint loaded from Hub

Task A [✓ CERTIFIED]  Sports      100.00%  "The national team won the championship after a stunning comeback "
Task B [✓ CERTIFIED]  Sci/Tech    100.00%  "Quarterly earnings beat analyst expectations driven by strong clo"
Task C [✓ CERTIFIED]  Sci/Tech    100.00%  "New quantum computing startup secures massive initial funding rou"

